In [ ]:
pip install pyomo


In [ ]:
!apt-get install -y glpk-utils


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
glpk-utils is already the newest version (5.0-1).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.


In [ ]:
from pyomo.environ import SolverFactory
print(SolverFactory('glpk').available())


True


**TSP Probleme**

*Brunch & bound ( manuelle)*

In [ ]:
import math

dist = [
    [0, 10, 15, 20],
    [10, 0, 35, 25],
    [15, 35, 0, 30],
    [20, 25, 30, 0]
]

n = 4
distnc = math.inf
bestpath = []

def tsp_branch_and_bound(current, visited , cost , path):
    global distnc, bestpath
    if len(path) == n:
        total = cost + dist[current][0]
        if total < distnc:
            distnc = total
            bestpath = path[:]
        return
    for next_city in range(n):
        if not visited[next_city] :
            new_cost = cost + dist[current][next_city]
            if new_cost < distnc:
                visited[next_city] = True
                path.append(next_city)
                tsp_branch_and_bound(next_city, visited, new_cost, path)
                path.pop()
                visited[next_city] = False

def solve_tsp():
    visited = [False] * n
    visited[0] = True
    tsp_branch_and_bound(0, visited, 0, [0])
    print("Meilleur chemin :", " -> ".join(chr(65+i) for i in bestpath + [bestpath[0]]))
    print("La distance total :", distnc)

solve_tsp()


Meilleur chemin : A -> B -> D -> C -> A
La distance total : 80



 *GLPK with subtours*

In [ ]:
from pyomo.environ import *

cities = ['A', 'B', 'C', 'D' ,'E','F']
n = len(cities)

dist = [
    [0, 34, 72, 18, 51, 29, 67],
    [34, 0, 55, 40, 22, 78, 60],
    [72, 55, 0, 63, 57, 30, 44],
    [18, 40, 63, 0, 50, 33, 21],
    [51, 22, 57, 50, 0, 43, 39],
    [29, 78, 30, 33, 43, 0, 54],
    [67, 60, 44, 21, 39, 54, 0]
]

I = range(n)
J = range(n)

model = ConcreteModel()

model.x = Var(I, J, domain=Binary)


model.obj = Objective(expr=sum(dist[i][j]*model.x[i,j] for i in I for j in J), sense=minimize)

model.constraints = ConstraintList()
for i in I:
    model.constraints.add(sum(model.x[i,j] for j in J if j != i) == 1)
    model.constraints.add(sum(model.x[j,i] for j in I if j != i) == 1)

solver = SolverFactory('glpk')
solver.solve(model)

tour = []
current = 0
for _ in range(n):
    for j in J:
        if model.x[current,j].value is not None and model.x[current,j].value > 0.5:
            tour.append(current)
            current = j
            break
tour.append(tour[0])

print(" -> ".join(cities[i] for i in tour))

total_cost = sum(dist[tour[i]][tour[i+1]] for i in range(len(tour)-1))
print("Coût total:", total_cost)


A -> D -> A -> D -> A -> D -> A
Coût total: 108


*GLPK without subtours (MTZ)*

In [ ]:
from pyomo.environ import *
import random


n = 50
dist = [[0]*n for _ in range(n)]

for i in range(n):
    for j in range(i+1, n):
        d = random.randint(10, 100)
        dist[i][j] = d
        dist[j][i] = d

for i in range(n):
    print(dist[i])

I = range(n)
J = range(n)

model = ConcreteModel()

model.x = Var(I, J, domain=Binary)
model.u = Var(I, domain=NonNegativeReals, bounds=(1,n-1))

model.obj = Objective(expr=sum(dist[i][j]*model.x[i,j] for i in I for j in J), sense=minimize)

model.constraints = ConstraintList()


for i in I:
    model.constraints.add(sum(model.x[i,j] for j in J if j != i) == 1)
    model.constraints.add(sum(model.x[j,i] for j in I if j != i) == 1)


for i in range(1,n):
    for j in range(1,n):
        if i != j:
            model.constraints.add(model.u[i] - model.u[j] + (n-1)*model.x[i,j] <= n-2)

solver = SolverFactory('glpk')
result = solver.solve(model, tee=False)


tour = []
current = 0

for _ in range(n):
    for j in J:
        if model.x[current,j].value is not None and model.x[current,j].value > 0.5:
            tour.append(current)
            current = j
            break

tour.append(tour[0])


path_str = " -> ".join(chr(65+i) for i in tour)
print("Meilleur chemin est :", path_str)


total_cost = sum(dist[tour[i]][tour[i+1]] for i in range(len(tour)-1))
print("Avec un coût total égal à :", total_cost)


[0, 98, 53, 55, 55, 49, 65, 38, 44, 59, 32, 90, 96, 25, 54, 66, 51, 31, 29, 70, 43, 94, 38, 68, 84, 91, 62, 69, 53, 43, 68, 91, 96, 16, 16, 85, 27, 70, 45, 58, 27, 13, 94, 62, 64, 75, 80, 26, 91, 57]
[98, 0, 13, 89, 47, 57, 23, 15, 76, 97, 51, 56, 36, 47, 88, 10, 20, 83, 89, 70, 92, 64, 69, 51, 63, 13, 68, 59, 65, 75, 85, 30, 96, 86, 24, 68, 52, 81, 28, 38, 86, 43, 74, 89, 33, 79, 86, 47, 61, 90]
[53, 13, 0, 75, 96, 63, 51, 10, 53, 44, 89, 99, 18, 90, 13, 37, 48, 67, 10, 53, 39, 20, 39, 14, 55, 11, 57, 52, 70, 73, 76, 92, 36, 98, 90, 20, 44, 52, 26, 27, 44, 40, 10, 51, 94, 11, 50, 85, 19, 17]
[55, 89, 75, 0, 38, 66, 24, 88, 40, 91, 58, 91, 59, 25, 91, 45, 88, 95, 27, 93, 51, 81, 25, 61, 79, 97, 79, 24, 36, 78, 58, 94, 66, 50, 38, 48, 57, 25, 57, 31, 33, 91, 65, 90, 65, 34, 74, 80, 62, 20]
[55, 47, 96, 38, 0, 95, 85, 61, 42, 18, 44, 37, 90, 20, 50, 92, 44, 24, 65, 70, 60, 80, 43, 74, 86, 80, 22, 96, 27, 62, 37, 45, 92, 96, 40, 100, 64, 56, 97, 87, 95, 36, 26, 33, 77, 63, 84, 35, 36, 41]

In [ ]:
from pyomo.environ import *
import random


n = 26
dist = [[0]*n for _ in range(n)]

for i in range(n):
    for j in range(i+1, n):
        d = random.randint(10, 100)
        dist[i][j] = d
        dist[j][i] = d

for i in range(n):
    print(dist[i])

I = range(n)
J = range(n)

model = ConcreteModel()

model.x = Var(I, J, domain=Binary)
model.u = Var(I, domain=NonNegativeReals, bounds=(1,n-1))

model.obj = Objective(expr=sum(dist[i][j]*model.x[i,j] for i in I for j in J), sense=minimize)

model.constraints = ConstraintList()


for i in I:
    model.constraints.add(sum(model.x[i,j] for j in J if j != i) == 1)
    model.constraints.add(sum(model.x[j,i] for j in I if j != i) == 1)


for i in range(1,n):
    for j in range(1,n):
        if i != j:
            model.constraints.add(model.u[i] - model.u[j] + (n-1)*model.x[i,j] <= n-2)

solver = SolverFactory('glpk')
result = solver.solve(model, tee=False)


tour = []
current = 0

for _ in range(n):
    for j in J:
        if model.x[current,j].value is not None and model.x[current,j].value > 0.5:
            tour.append(current)
            current = j
            break

tour.append(tour[0])


path_str = " -> ".join(chr(65+i) for i in tour)
print("Meilleur chemin est :", path_str)


total_cost = sum(dist[tour[i]][tour[i+1]] for i in range(len(tour)-1))
print("Avec un coût total égal à :", total_cost)


[0, 100, 97, 69, 85, 46, 16, 47, 41, 11, 92, 64, 88, 39, 71, 31, 76, 66, 63, 39, 40, 36, 23, 42, 80, 37]
[100, 0, 37, 92, 41, 84, 37, 91, 44, 21, 77, 21, 37, 26, 97, 30, 70, 64, 28, 34, 80, 44, 14, 76, 14, 24]
[97, 37, 0, 56, 76, 66, 59, 68, 34, 76, 85, 46, 65, 97, 88, 73, 27, 38, 38, 91, 90, 19, 42, 77, 67, 77]
[69, 92, 56, 0, 48, 72, 99, 80, 46, 28, 80, 34, 61, 27, 77, 80, 94, 89, 80, 10, 25, 27, 48, 41, 24, 19]
[85, 41, 76, 48, 0, 35, 54, 86, 19, 31, 48, 74, 91, 86, 30, 22, 21, 59, 32, 98, 85, 75, 25, 74, 33, 82]
[46, 84, 66, 72, 35, 0, 75, 58, 16, 12, 100, 14, 52, 77, 98, 34, 19, 98, 92, 37, 24, 72, 67, 55, 56, 20]
[16, 37, 59, 99, 54, 75, 0, 24, 10, 36, 89, 65, 57, 73, 47, 31, 38, 23, 71, 26, 56, 21, 64, 64, 81, 11]
[47, 91, 68, 80, 86, 58, 24, 0, 48, 78, 77, 100, 22, 44, 32, 42, 17, 92, 85, 85, 92, 21, 47, 41, 85, 42]
[41, 44, 34, 46, 19, 16, 10, 48, 0, 84, 12, 32, 95, 91, 48, 79, 28, 53, 19, 50, 49, 90, 45, 63, 38, 87]
[11, 21, 76, 28, 31, 12, 36, 78, 84, 0, 51, 25, 90, 74, 15, 

In [ ]:
from pyomo.environ import *
import random


n = 5
dist = [[0]*n for _ in range(n)]

for i in range(n):
    for j in range(i+1, n):
        d = random.randint(10, 100)
        dist[i][j] = d
        dist[j][i] = d

for i in range(n):
    print(dist[i])

I = range(n)
J = range(n)

model = ConcreteModel()

model.x = Var(I, J, domain=Binary)
model.u = Var(I, domain=NonNegativeReals, bounds=(1,n-1))

model.obj = Objective(expr=sum(dist[i][j]*model.x[i,j] for i in I for j in J), sense=minimize)

model.constraints = ConstraintList()


for i in I:
    model.constraints.add(sum(model.x[i,j] for j in J if j != i) == 1)
    model.constraints.add(sum(model.x[j,i] for j in I if j != i) == 1)


for i in range(1,n):
    for j in range(1,n):
        if i != j:
            model.constraints.add(model.u[i] - model.u[j] + (n-1)*model.x[i,j] <= n-2)

solver = SolverFactory('glpk')
result = solver.solve(model, tee=True)


tour = []
current = 0

for _ in range(n):
    for j in J:
        if model.x[current,j].value is not None and model.x[current,j].value > 0.5:
            tour.append(current)
            current = j
            break

tour.append(tour[0])


path_str = " -> ".join(chr(65+i) for i in tour)
print("Meilleur chemin est :", path_str)


total_cost = sum(dist[tour[i]][tour[i+1]] for i in range(len(tour)-1))
print("Avec un coût total égal à :", total_cost)


[0, 30, 83, 66, 11]
[30, 0, 41, 96, 19]
[83, 41, 0, 22, 56]
[66, 96, 22, 0, 55]
[11, 19, 56, 55, 0]
GLPSOL--GLPK LP/MIP Solver 5.0
Parameter(s) specified in the command line:
 --write /tmp/tmpzfm04g00.glpk.raw --wglp /tmp/tmpv94imi2j.glpk.glp --cpxlp
 /tmp/tmp0eqhzxjl.pyomo.lp
Reading problem data from '/tmp/tmp0eqhzxjl.pyomo.lp'...
/tmp/tmp0eqhzxjl.pyomo.lp:196: warning: lower bound of variable 'x2' redefined
/tmp/tmp0eqhzxjl.pyomo.lp:196: warning: upper bound of variable 'x2' redefined
22 rows, 24 columns, 76 non-zeros
20 integer variables, all of which are binary
216 lines were read
Writing problem data to '/tmp/tmpv94imi2j.glpk.glp'...
171 lines were written
GLPK Integer Optimizer 5.0
22 rows, 24 columns, 76 non-zeros
20 integer variables, all of which are binary
Preprocessing...
22 rows, 24 columns, 76 non-zeros
20 integer variables, all of which are binary
Scaling...
 A: min|aij| =  1.000e+00  max|aij| =  4.000e+00  ratio =  4.000e+00
Problem data seem to be well scaled
Construct